In [1]:
import os
import time
import glob
import pandas as pd
import numpy  as np
import tables as tb

import matplotlib
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.cm     as colormap

from mpl_toolkits               import mplot3d
from mpl_toolkits.mplot3d       import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# cities
from invisible_cities.cities.detsim      import detsim
from invisible_cities.cities.hypathia    import hypathia
from invisible_cities.cities.penthesilea import penthesilea
from invisible_cities.cities.esmeralda   import esmeralda
from invisible_cities.cities.beersheba   import beersheba

from invisible_cities.core               import system_of_units as units
pes = units.pes
ns, mus, ms = units.ns, units.mus, units.ms
eV, keV = units.eV, units.keV
mm, cm  = units.mm, units.cm
hertz   = units.hertz
kV = units.kV
bar = units.bar
from invisible_cities.core.configure   import configure
from invisible_cities.core.configure   import all             as all_events

from invisible_cities.database import load_db

In [2]:
plt.rcParams["font.size"]      = 15
plt.rcParams["font.family"]    = "sans-serif"
plt.rcParams["font.monospace"] = "Computer Modern Sans Serif"

In [3]:
inputfilename = os.path.expandvars("$HOME/NEXT/DATA/nexus/nexus_0_0nubb.h5")

# DetSim

**El-gain**: \
https://www.sciencedirect.com/science/article/pii/S0370269310000420 \
https://core.ac.uk/reader/82436370 \
\
**Diffusion**: \
https://arxiv.org/abs/1804.01680

$D'_{L, T} = \frac{D^*_{L, T}}{\sqrt{P}}$

In [4]:
E  = (7/0.5) * kV/cm
p  = 8.15    * bar
dx = 0.5     * cm
el_gain = dx*p*(170*(1/kV)*(E/p - 0.70*(kV/cm/bar)))

transverse_diffusion   = 1.2 * mm/cm**0.5
longitudinal_diffusion = 0.3 * mm/cm**0.5

In [8]:
# configure
conf = configure('detsim $ICTDIR/invisible_cities/config/detsim.conf'.split())

conf["files_in"]    = inputfilename
conf["file_out"]    = "/tmp/detsim.h5"
conf["event_range"] = (5, 6)
conf["detector_db"] = "next100"
conf["run_number"]  = 0
conf["s1_lighttable"] = "$HOME/NEXT/DATA/LightTables/NEXT100_S1_LT.h5"
conf["s2_lighttable"] = "$HOME/NEXT/DATA/LightTables/NEXT100_S2_LT.h5"
conf["sipm_psf"]      = "$HOME/NEXT/DATA/LightTables/NEXT100_PSF.h5"

conf["physics_params"] = {"ws": 39.2 * eV
                         ,"wi": 22.4 * eV
                         ,"fano_factor"           : 0.15
                         ,"conde_policarpo_factor": 1.0
                         ,"drift_velocity"        : 1.0 * mm/mus
                         ,"lifetime"              : 12  * ms
                         ,"transverse_diffusion"  : transverse_diffusion
                         ,"longitudinal_diffusion": longitudinal_diffusion
                         ,"el_gain"               : el_gain
                         ,"el_drift_velocity"     : 2.5 * mm/mus}

conf["buffer_params"]  = {"length"     : 800 * mus
                         ,"pmt_width"  :  25 * ns 
                         ,"sipm_width" :   1 * mus
                         ,"max_time"   :   1 * ms
                         ,"pre_trigger":  10 * mus
                         ,"trigger_thr": 0}

conf["print_mod"] = 1
conf["rate"]      = 0.5 * hertz

dconf = conf # save detsim conf

# run
t0 = time.time()
result = detsim(**conf)
print("Exec (s)", time.time()-t0)

events processed: 0, event number: 5
Exec (s) 16.43758988380432


In [9]:
event = 0
with tb.open_file(dconf["file_out"]) as h5file:
    pmtrd  = h5file.root.pmtrd [event]
    sipmrd = h5file.root.sipmrd[event]

In [10]:
plt.figure()
times = np.arange(0, dconf["buffer_params"]["length"], dconf["buffer_params"]["pmt_width"])/mus
plt.scatter(times, pmtrd.sum(axis=0), s=2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [11]:
datasipm = load_db.DataSiPM(dconf["detector_db"], dconf["run_number"])

charge = np.sum(sipmrd, axis=1)
sel = charge >= 0
norm   = colors.Normalize(vmin=min(charge), vmax=max(charge), clip=True)
mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

plt.figure()
plt.scatter(datasipm["X"][sel], datasipm["Y"][sel], marker=".", s=20, color=mapper.to_rgba(charge[sel]))
plt.colorbar(mapper)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

locator: <matplotlib.colorbar._ColorbarAutoLocator object at 0x7ff7cfebfb90>
Using auto colorbar locator on colorbar
locator: <matplotlib.colorbar._ColorbarAutoLocator object at 0x7ff7cfebfb90>
Setting pcolormesh


# Hypathia

In [12]:
# configure
conf = configure('hypathia $ICTDIR/invisible_cities/config/hypathia.conf'.split())

conf["files_in"]   = "/tmp/detsim.h5"
conf["file_out"]   = "/tmp/hypathia.h5"
conf["detector_db"]= "next100"
conf["run_number"] = 0
conf["event_range"]= all_events
conf["print_mod"]  = 1

conf["pmt_wfs_rebin"]  = 1
conf["pmt_pe_rms"]     = 0
conf["sipm_noise_cut"] = 0
conf["filter_padding"] = 50

conf["thr_csum_s1"]   = 0.5 * pes
conf["thr_csum_s2"]   = 2.0 * pes
conf["thr_sipm"]      = 1.0 * pes
conf["thr_sipm_type"] = "common"

conf["s1_tmin"]   = 0  * mus
conf["s1_tmax"]   = 11 * mus
conf["s1_stride"] = 4 
conf["s1_lmin"]   = 4
conf["s1_lmax"]   = 40
conf["s1_rebin_stride"] = 1

conf["s2_tmin"]   = 11  * mus
conf["s2_tmax"]   = 800 * mus
conf["s2_stride"] = 40
conf["s2_lmin"]   = 80
conf["s2_lmax"]   = 100000
conf["s2_rebin_stride"] = 40

conf["thr_sipm_s2"] = 1.0 * pes

hconf = conf

# run
t0 = time.time()
result = hypathia(**conf)
print("Exec (s)", time.time()-t0)

events processed: 0, event number: 10
Exec (s) 7.217766761779785


In [13]:
S1   = pd.read_hdf(hconf["file_out"], "PMAPS/S1")
S2   = pd.read_hdf(hconf["file_out"], "PMAPS/S2")
S2Si = pd.read_hdf(hconf["file_out"], "PMAPS/S2Si")

In [14]:
plt.figure()
times = np.arange(0, dconf["buffer_params"]["length"], dconf["buffer_params"]["pmt_width"])/mus
plt.scatter(times, pmtrd.sum(axis=0)/pmtrd.sum(axis=0).max(), s=2, color="k")

# S1
plt.scatter(S1["time"]/mus, S1["ene"]/pmtrd.sum(axis=0).max(), s=20, facecolor="none", edgecolor="b", label="S1")

# S2
plt.scatter(S2["time"]/mus, S2["ene"]/S2["ene"].max(), s=20, facecolor="none", edgecolor="r", label="S2")

plt.legend(markerscale=2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [15]:
datasipm = load_db.DataSiPM(conf["detector_db"], conf["run_number"])

charge   = S2Si.groupby("nsipm").ene.sum().to_frame()
sel_sipm = datasipm.loc[charge.index]
norm   = colors.Normalize(vmin=min(charge["ene"]), vmax=max(charge["ene"]), clip=True)
mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

plt.figure()
plt.scatter(sel_sipm["X"], sel_sipm["Y"], marker=".", s=20, color=mapper.to_rgba(charge["ene"]))
plt.colorbar(mapper)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

locator: <matplotlib.colorbar._ColorbarAutoLocator object at 0x7ff7c96f5550>
Using auto colorbar locator on colorbar
locator: <matplotlib.colorbar._ColorbarAutoLocator object at 0x7ff7c96f5550>
Setting pcolormesh


# Penthesilea

In [16]:
# configure
conf = configure('penthesilea $ICTDIR/invisible_cities/config/penthesilea.conf'.split())

conf["files_in"]   = "/tmp/hypathia.h5"
conf["file_out"]   = "/tmp/penthesilea.h5"
conf["detector_db"]= "next100"
conf["run_number"] = 0
conf["event_range"]= all_events
conf["print_mod"]  = 1

conf["s1_nmin"] = 1
conf["s1_nmax"] = 5
conf["s1_emin"] = 0   * pes
conf["s1_emax"] = 1e3 * pes
conf["s1_wmin"] = 25  * ns
conf["s1_wmax"] =  1  * mus
conf["s1_hmin"] =  0  * pes
conf["s1_hmax"] = 1e3 * pes
conf["s1_ethr"] =   0 * pes

conf["s2_nmin"] = 1
conf["s2_nmax"] = 5
conf["s2_emin"] =   0 * pes
conf["s2_emax"] = 1e6 * pes
conf["s2_wmin"] =   1 * mus
conf["s2_wmax"] = 1e6 * mus
conf["s2_hmin"] =   0 * pes
conf["s2_hmax"] = 1e6 * pes
conf["s2_ethr"] =   0 * pes
conf["s2_nsipmmin"] = 1
conf["s2_nsipmmax"] = 1e5

conf["drift_v"] = 1.0 * mm/mus
conf["rebin"]   = 1

conf["slice_reco_params"]  = dict( Qthr          =  5 * pes
                                 , Qlm           =  0 * pes 
                                 , lm_radius     = 0.001 * mm
                                 , new_lm_radius = 0.001 * mm 
                                 , msipm         =  1)

conf["global_reco_params"] = dict( Qthr          = 20 * pes
                                 , Qlm           =  0 * pes
                                 , lm_radius     = -1 * mm
                                 , new_lm_radius = -1 * mm
                                 , msipm         =  1)

pconf = conf

# run
t0 = time.time()
result = penthesilea(**conf)
print("Exec (s)", time.time()-t0)

events processed: 0, event number: 10
Exec (s) 1.7356107234954834


In [17]:
DST    = pd.read_hdf(pconf["file_out"], "DST/Events")
RECO   = pd.read_hdf(pconf["file_out"], "RECO/Events")
mchits = pd.read_hdf(pconf["file_out"], "MC/hits")

In [18]:
fig = plt.figure()
ax = fig.gca(projection='3d')

norm   = colors  .Normalize(vmin=0, vmax=RECO["Q"].max(), clip=True)
mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

# slice reco
ax.scatter(RECO["X"], RECO["Y"], RECO["Z"], color=mapper.to_rgba(RECO["Q"]), alpha=0.05)
# global reco
ax.scatter(DST["X"], DST["Y"], DST["Z"], marker="*", facecolor="w", edgecolor="r", s=100, alpha=1)

# mchits
ax.scatter(mchits["x"], mchits["y"], mchits["z"], color="r", s=2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

# Esmeralda

In [19]:
# configure
conf = configure('esmeralda $ICTDIR/invisible_cities/config/esmeralda.conf'.split())

conf["files_in"]    = "/tmp/penthesilea.h5"
conf["file_out"]    = "/tmp/esmeralda.h5"
conf["run_number"]  = 0
conf["detector_db"] = "next100"
conf["event_range"] = all_events
conf["print_mod"]   = 1

conf["cor_hits_params"] = dict( map_fname = "$HOME/NEXT/DATA/LightTables/map_NEXT100_detsim.h5"
                              , threshold_charge_low  =  5 * pes
                              , threshold_charge_high =  5 * pes
                              , same_peak             = True
                              , norm_strat            = "kr"
                              , apply_temp            = False)

conf["paolina_params"] = dict( vox_size         = [15 * mm, 15 * mm, 15 * mm]
                             , strict_vox_size  = False
                             , energy_threshold = 10 * keV
                             , min_voxels       = 3
                             , blob_radius      = 21 * mm
                             , max_num_hits     = 30000)

econf = conf

t0 = time.time()
result = esmeralda(**conf)
print("Exec (s)", time.time()-t0)

events processed: 0, event number: 10
Exec (s) 45.85487484931946


In [20]:
CHITS   = pd.read_hdf(econf["file_out"], "CHITS/highTh")
Tracks  = pd.read_hdf(econf["file_out"], "Tracking/Tracks")
Summary = pd.read_hdf(econf["file_out"], "Summary/Events")
DST     = pd.read_hdf(econf["file_out"], "DST/Events")
MCHITS  = pd.read_hdf(econf["file_out"], "MC/hits")

In [21]:
chits  = CHITS
tracks = Tracks
mchits = MCHITS

# clear spureous tracks
tracks = tracks[tracks["length"]>0]

In [22]:
fig = plt.figure()
ax = fig.gca(projection='3d')

for track_id, track in tracks.groupby("trackID"):
    hits = chits[chits["track_id"]==track_id]
    
    # plot
    norm   = colors.Normalize(vmin=0, vmax=hits["E"].max(), clip=True)
    mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

    # chits
    ax.scatter(hits["X"], hits["Y"], hits["Z"], color=mapper.to_rgba(hits["E"]), alpha=0.1)

    # mchits
    ax.scatter(mchits["x"], mchits["y"], mchits["z"], c="r", s=1)

    # blobs
    ax.scatter(track["blob1_x"], track["blob1_y"], track["blob1_z"], marker="*", s=200, c="k")
    ax.scatter(track["blob2_x"], track["blob2_y"], track["blob2_z"], marker="*", s=200, c="k")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [23]:
# from invisible_cities.io.hits_io       import hits_from_df
# from invisible_cities.cities.esmeralda import track_blob_info_creator_extractor

# paolina_algorithm = track_blob_info_creator_extractor(**conf["paolina_params"])

# hitc = hits_from_df(chits)[chits["event"].unique()[0]]
# df, track_hitc, out_of_map = paolina_algorithm(hitc)

# Beersheba

In [24]:
# configure
conf = configure('beersheba $ICTDIR/invisible_cities/config/beersheba.conf'.split())

conf["files_in"]    = "/tmp/esmeralda.h5"
conf["file_out"]    = "/tmp/beersheba.h5"
conf["run_number"]  = 0
conf["detector_db"] = "next100"
conf["event_range"] = all_events
conf["print_mod"]   = 1

conf["deconv_params"] = dict( q_cut         = 10
                            , drop_dist     = [16., 16.]
                            , psf_fname     = "$HOME/NEXT/DATA/LightTables/next100.kr83m_202103.psf.h5"
                            , e_cut         = 8e-3
                            , n_iterations  = 100
                            , iteration_tol = 1e-10
                            , sample_width  = [15.55, 15.55]
                            , bin_size      = [ 1.,  1.]
                            , energy_type   = "Ec"
                            , diffusion     = (1.0, 0.2)
                            , deconv_mode   = "joint"
                            , n_dim         = 2
                            , cut_type      = "abs"
                            , inter_method  = "cubic")

bconf = conf

t0 = time.time()
result = beersheba(**conf)
print("Exec (s)", time.time()-t0)

events processed: 0, event number: 10
Exec (s) 99.71186423301697


In [25]:
DECO = pd.read_hdf(bconf["file_out"], "DECO/Events")

In [26]:
fig = plt.figure()
ax = fig.gca(projection='3d')

norm   = colors  .Normalize(vmin=0, vmax=DECO["E"].max(), clip=True)
mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

ax.scatter(DECO["X"], DECO["Y"], DECO["Z"], color=mapper.to_rgba(DECO["E"]), alpha=0.1)

# mchits
ax.scatter(mchits["x"], mchits["y"], mchits["z"], c="r", s=1)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [778]:
# def track_blob_info_creator_extractor(vox_size : [float, float, float], energy_type : evm.HitEnergy, strict_vox_size : bool, energy_threshold : float, min_voxels : int, blob_radius : float) -> Callable:
#   """ Wrapper of extract_track_blob_info"""
#   def create_extract_track_blob_info(hitc):
    
#     voxels   = plf.voxelize_hits(hitc.hits, vox_size, strict_vox_size, energy_type)
#     mod_voxels = voxels if energy_threshold == 0. else plf.drop_end_point_voxels(voxels, energy_threshold, min_voxels)
#     tracks   = plf.make_track_graphs(mod_voxels)
#     vox_size_x = voxels[0].size[0]
#     vox_size_y = voxels[0].size[1]
#     vox_size_z = voxels[0].size[2]
#     #sort tracks in energy
#     tracks   = sorted(tracks, key = lambda t: sum([vox.Ehits for vox in t.nodes()]), reverse = True)
#     track_hits = []
#     df = pd.DataFrame(columns=['event', 'trackID', 'energy', 'length', 'numb_of_voxels',
#                   'numb_of_hits', 'numb_of_tracks', 'x_min', 'y_min', 'z_min',
#                   'x_max', 'y_max', 'z_max', 'r_max', 'x_ave', 'y_ave', 'z_ave',
#                   'extreme1_x', 'extreme1_y', 'extreme1_z',
#                   'extreme2_x', 'extreme2_y', 'extreme2_z',
#                   'blob1_x', 'blob1_y', 'blob1_z',
#                   'blob2_x', 'blob2_y', 'blob2_z',
#                   'eblob1', 'eblob2', 'ovlp_blob_energy',
#                   'vox_size_x', 'vox_size_y', 'vox_size_z'])
#     for c, t in enumerate(tracks, 0):
#       tID = c
#       energy = sum([vox.Ehits for vox in t.nodes()])
#       length = plf.length(t)
#       numb_of_hits = len([h for vox in t.nodes() for h in vox.hits])
#       numb_of_voxels = len(t.nodes())
#       numb_of_tracks = len(tracks  )
#       min_x = min([h.X for v in t.nodes() for h in v.hits])
#       max_x = max([h.X for v in t.nodes() for h in v.hits])
#       min_y = min([h.Y for v in t.nodes() for h in v.hits])
#       max_y = max([h.Y for v in t.nodes() for h in v.hits])
#       min_z = min([h.Z for v in t.nodes() for h in v.hits])
#       max_z = max([h.Z for v in t.nodes() for h in v.hits])
#       max_r = max([np.sqrt(h.X*h.X + h.Y*h.Y) for v in t.nodes() for h in v.hits])
#       pos = [h.pos for v in t.nodes() for h in v.hits]
#       e  = [getattr(h, energy_type.value) for v in t.nodes() for h in v.hits]
#       ave_pos = np.average(pos, weights=e, axis=0)
#       extr1, extr2 = plf.find_extrema(t)
#       extr1_pos = extr1.XYZ
#       extr2_pos = extr2.XYZ
#       blob_pos1, blob_pos2 = plf.blob_centres(t, blob_radius)
#       e_blob1, e_blob2, hits_blob1, hits_blob2 = plf.blob_energies_and_hits(t, blob_radius)
#       overlap = False
#       overlap = sum([h.Ec for h in set(hits_blob1).intersection(hits_blob2)])
#       list_of_vars = [hitc.event, tID, energy, length, numb_of_voxels, numb_of_hits, numb_of_tracks, min_x, min_y, min_z, max_x, max_y, max_z, max_r, ave_pos[0], ave_pos[1], ave_pos[2], extr1_pos[0], extr1_pos[1], extr1_pos[2], extr2_pos[0], extr2_pos[1], extr2
# _pos[2], blob_pos1[0], blob_pos1[1], blob_pos1[2], blob_pos2[0], blob_pos2[1], blob_pos2[2], e_blob1, e_blob2, overlap, vox_size_x, vox_size_y, vox_size_z]
#       df.loc[c] = list_of_vars
#       try:
#         types_dict
#       except NameError:
#         types_dict = dict(zip(df.columns, [type(x) for x in list_of_vars]))
# for vox in t.nodes():
#         for hit in vox.hits:
#           hit.track_id = tID
#           track_hits.append(hit)
#     track_hitc = evm.HitCollection(hitc.event, hitc.time)
#     track_hitc.hits = track_hits
#     #change dtype of columns to match type of variables
#     df = df.apply(lambda x : x.astype(types_dict[x.name]))
#     return df, mod_voxels, track_hitc
#   return create_extract_track_blob_info

In [30]:
from invisible_cities.types.ic_types import xy
from invisible_cities.evm            import event_model       as evm
from invisible_cities.reco           import paolina_functions as plf

from invisible_cities.cities.esmeralda import types_dict_tracks

def track_blob_info_creator_extractor(vox_size         : [float, float, float],
                                      strict_vox_size  : bool                 ,
                                      energy_threshold : float                ,
                                      min_voxels       : int                  ,
                                      blob_radius      : float                ,
                                      max_num_hits     : int):
    def create_extract_track_blob_info(hitc):
        df = pd.DataFrame(columns=list(types_dict_tracks.keys()))
        if len(hitc.hits) > max_num_hits:
            return df, hitc, True
        #track_hits is a new Hitcollection object that contains hits belonging to tracks, and hits that couldnt be corrected
        track_hitc = evm.HitCollection(hitc.event, hitc.time)
        out_of_map = np.any(np.isnan([h.Ep for h in hitc.hits]))
        if out_of_map:
            #add nan hits to track_hits, the track_id will be -1
            track_hitc.hits.extend  ([h for h in hitc.hits if np.isnan   (h.Ep)])
            hits_without_nan       = [h for h in hitc.hits if np.isfinite(h.Ep)]
            #create new Hitcollection object but keep the name hitc
            hitc      = evm.HitCollection(hitc.event, hitc.time)
            hitc.hits = hits_without_nan

        if len(hitc.hits) > 0:
            voxels           = plf.voxelize_hits(hitc.hits, vox_size, strict_vox_size, evm.HitEnergy.Ep)
#             (    mod_voxels,
#              dropped_voxels) = plf.drop_end_point_voxels(voxels, energy_threshold, min_voxels)
            mod_voxels = voxels
            tracks           = plf.make_track_graphs(mod_voxels)

#             for v in dropped_voxels:
#                 track_hitc.hits.extend(v.hits)

            vox_size_x = voxels[0].size[0]
            vox_size_y = voxels[0].size[1]
            vox_size_z = voxels[0].size[2]
            del(voxels)
            #sort tracks in energy
            tracks     = sorted(tracks, key=plf.get_track_energy, reverse=True)

            track_hits = []
            for c, t in enumerate(tracks, 0):
                tID = c
                energy = plf.get_track_energy(t)
                length = plf.length(t)
                numb_of_hits   = len([h for vox in t.nodes() for h in vox.hits])
                numb_of_voxels = len(t.nodes())
                numb_of_tracks = len(tracks   )
                pos   = [h.pos for v in t.nodes() for h in v.hits]
                x, y, z = map(np.array, zip(*pos))
                r = np.sqrt(x**2 + y**2)

                e     = [h.Ep for v in t.nodes() for h in v.hits]
                ave_pos = np.average(pos, weights=e, axis=0)
                ave_r   = np.average(r  , weights=e, axis=0)
                extr1, extr2 = plf.find_extrema(t)
                extr1_pos = extr1.XYZ
                extr2_pos = extr2.XYZ

                blob_pos1, blob_pos2 = plf.blob_centres(t, blob_radius)

                e_blob1, e_blob2, hits_blob1, hits_blob2 = plf.blob_energies_and_hits(t, blob_radius)
                overlap = float(sum(h.Ep for h in set(hits_blob1).intersection(set(hits_blob2))))
                list_of_vars = [hitc.event, tID, energy, length, numb_of_voxels,
                                numb_of_hits, numb_of_tracks,
                                min(x), min(y), min(z), min(r), max(x), max(y), max(z), max(r),
                                *ave_pos, ave_r, *extr1_pos,
                                *extr2_pos, *blob_pos1, *blob_pos2,
                                e_blob1, e_blob2, overlap,
                                vox_size_x, vox_size_y, vox_size_z]

                df.loc[c] = list_of_vars

                for vox in t.nodes():
                    for hit in vox.hits:
                        hit.track_id = tID
                        track_hits.append(hit)

            #change dtype of columns to match type of variables
            df = df.apply(lambda x : x.astype(types_dict_tracks[x.name]))
            track_hitc.hits.extend(track_hits)
        return df, mod_voxels, track_hitc, out_of_map

    return create_extract_track_blob_info

def convert_df_to_hits(df):
    return [evm.Hit(0, evm.Cluster(0, xy(h.X,h.Y), xy(0,0), 0), h.Z, h.E, xy(0, 0))
           for h in df.itertuples(index=False)]

In [28]:
# from invisible_cities.io.hits_io       import hits_from_df
# from invisible_cities.cities.esmeralda import track_blob_info_creator_extractor

# esmeralda hits
# paolina_algorithm = track_blob_info_creator_extractor(**econf["paolina_params"])
# hitc = hits_from_df(chits)[chits["event"].unique()[0]]

In [31]:
paolina_params = dict( vox_size         = [15 * mm, 15 * mm, 15 * mm]
                     , strict_vox_size  = False
                     , energy_threshold = 10 * keV
                     , min_voxels       = 3
                     , blob_radius      = 21 * mm
                     , max_num_hits     = 30000)

paolina_algorithm = track_blob_info_creator_extractor(**paolina_params)

# deco hits
event = DECO["event"].unique()[0]
hits  = convert_df_to_hits(DECO)
hitc  = evm.HitCollection(event, 0, hits)

df, voxels, track_hitc, out_of_map = paolina_algorithm(hitc)

In [32]:
# select track ids with length>0
df = df[df["length"]>0]
track_ids = df.trackID.values

voxels = [voxel for voxel in voxels if voxel.hits[0].track_id in track_ids]

In [36]:
voxel_pos  = np.array([voxel.pos for voxel in voxels])
voxel_ene  = np.array([voxel.E   for voxel in voxels])
# voxel_size = np.unique([voxel.size for voxel in voxels])

# x = np.arange(min(voxel_pos[:, 0])-voxel_size[0]/2., max(voxel_pos[:, 0]) + voxel_size[0]/2., voxel_size[0])
# y = np.arange(min(voxel_pos[:, 1])-voxel_size[1]/2., max(voxel_pos[:, 1]) + voxel_size[1]/2., voxel_size[1])
# z = np.arange(min(voxel_pos[:, 2])-voxel_size[2]/2., max(voxel_pos[:, 2]) + voxel_size[2]/2., voxel_size[2])

lower_corners = np.array([voxel.pos-voxel.size/2. for voxel in voxels])
upper_corners = np.array([voxel.pos+voxel.size/2. for voxel in voxels])
corners = np.concatenate((lower_corners, upper_corners))
corners = np.unique(corners, axis=0)
x, y, z = np.unique(corners[:, 0]), np.unique(corners[:, 1]), np.unique(corners[:, 2])

filled, _ = np.histogramdd(voxel_pos, bins=[x, y, z])
filled  = np.swapaxes(filled, 0, 1).astype(bool)
x, y, z = np.meshgrid(x, y, z)

In [37]:
fig = plt.figure()
ax = fig.gca(projection='3d')

norm   = colors  .Normalize(vmin=min(voxel_ene), vmax=max(voxel_ene), clip=True)
mapper = colormap.ScalarMappable(norm=norm, cmap=colormap.coolwarm)

ax.voxels(x, y, z, filled, alpha=0.5, color=mapper.to_rgba(voxel_ene))

# mchits
ax.scatter(mchits["x"], mchits["y"], mchits["z"], c="r", s=1)

# blobs
ax.scatter(df["blob1_x"], df["blob1_y"], df["blob1_z"], marker="*", s=200, c="k")
ax.scatter(df["blob2_x"], df["blob2_y"], df["blob2_z"], marker="*", s=200, c="k")

ax.view_init(30, 60)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …